Route Profitability Analysis (Gold Layer)

This notebook identifies underperforming routes, analyzes cost drivers, and evaluates performance trends across 3 years using Delta Lake data.


Define Underperforming Route

A route is considered underperforming if:

    Gross margin < 15%
    OR gross profit < 0
    OR operational inefficiencies exist (low utilization or completion rate)


In [0]:

from pyspark.sql.functions import *
df = spark.table("`gfl-gold`.`route_profitability_summary`")

underperforming_df = df.filter(
    (col("gross_margin_pct") < 15) |
    (col("total_profit") < 0)
)

display(underperforming_df)

date,year,month,quarter,region,bu,area,total_routes,total_stops,completed_stops,missed_stops,total_revenue,total_fuel_cost,total_labour_cost,total_disposal_cost,total_maintenance_cost,total_admin_cost,total_cost,total_profit,avg_stop_completion_rate,avg_route_utilization_pct,avg_delay_minutes,avg_revenue_per_km,avg_fuel_efficiency,avg_labour_efficiency,underperforming_routes,delayed_routes,gross_margin_pct,cost_per_stop,profit_per_stop
2024-12-04,2024,12,Q4,Alberta,Calgary BU,Calgary SE,1,143,143,0,4696.84,182.59,710.36,3249.08,68.76,285.37,4496.16,200.68,1.0,100.81577158395649,7.0,null,null,9.476474486414844,1,1,4.27,31.44,1.4
2024-12-28,2024,12,Q4,Ontario,Central Ontario BU,Kingston,1,163,159,4,6846.14,198.29,784.59,4859.15,71.03,181.72,6094.78,751.36,0.9754601226993865,104.54545454545456,57.0,null,null,7.482352941176471,1,1,10.97,37.39,4.73
2024-12-19,2024,12,Q4,Ontario,Central Ontario BU,Kingston,1,176,175,1,7367.09,182.8,932.81,4856.06,75.67,253.73,6301.07,1066.02,0.9943181818181818,100.0,0.0,null,null,7.612005219660722,1,0,14.47,35.8,6.09
2024-12-06,2024,12,Q4,Alberta,Calgary BU,Calgary SE,1,139,139,0,4823.11,196.93,879.41,3699.06,81.53,278.57,5135.5,-312.39,1.0,100.0,0.0,null,null,6.332574031890661,1,0,-6.48,36.95,-2.25
2024-12-03,2024,12,Q4,Quebec,Quebec BU,Laval,1,141,141,0,4371.66,282.41,822.26,4145.21,75.11,117.2,5442.19,-1070.53,1.0,100.0,0.0,null,null,7.930258717660292,1,0,-24.49,38.6,-7.59
2024-12-06,2024,12,Q4,Ontario,GTA BU,Toronto East,1,135,135,0,6381.21,170.74,786.78,4020.04,92.84,356.87,5427.27,953.94,1.0,100.2723311546841,3.0,null,null,8.07899461400359,1,1,14.95,40.2,7.07
2024-12-18,2024,12,Q4,Prairies,Prairies BU,Regina,1,115,115,0,4655.13,136.35,666.85,3297.62,44.89,131.04,4276.75,378.38,1.0,102.63340263340264,23.0,null,null,7.718120805369128,1,1,8.13,37.19,3.29
2024-12-23,2024,12,Q4,Prairies,Prairies BU,Regina,1,119,119,0,4544.05,229.28,736.91,3851.13,93.06,120.12,5030.5,-486.45,1.0,100.0,0.0,null,null,8.056872037914692,1,0,-10.71,42.27,-4.09
2024-12-18,2024,12,Q4,Alberta,Calgary BU,Airdrie,1,147,147,0,5188.08,289.87,860.16,3433.78,105.05,109.91,4798.77,389.31,1.0,100.0,0.0,null,null,7.223587223587223,1,0,7.5,32.64,2.65
2024-12-12,2024,12,Q4,Prairies,Prairies BU,Regina,1,112,106,6,3189.46,137.76,706.98,2514.62,89.98,149.08,3598.42,-408.96,0.9464285714285714,101.93798449612403,18.0,null,null,6.374022850270595,1,1,-12.82,32.13,-3.86


2. Underperforming Areas
We analyze which areas have the highest concentration of low-margin routes.

In [0]:
display(
    df.filter(col("gross_margin_pct") < 15)
      .groupBy("area")
      .agg(
          count("*").alias("underperforming_days"),
          avg("gross_margin_pct").alias("avg_margin")
      )
      .orderBy("underperforming_days", ascending=False)
)

area,underperforming_days,avg_margin
Fredericton,121,-6.901074380165289
Laval,111,-1.4600900900900897
Airdrie,103,-3.4132038834951457
Moncton,82,-2.017560975609756
St. Albert,80,-23.187875
Regina,73,-2.933424657534246
Kingston,59,0.028983050847457552
Barrie,55,2.5712727272727265
Abbotsford,52,5.431923076923076
Toronto East,40,-0.6829999999999998


In [0]:
### 3. Underperforming Business Units
display(
    df.groupBy("bu")
      .agg(
          avg("gross_margin_pct").alias("avg_margin"),
          sum("underperforming_routes").alias("underperforming_routes")
      )
      .orderBy("avg_margin")
)

bu,avg_margin,underperforming_routes
Calgary BU,32.63053491827638,146
Atlantic BU,42.530023966446976,431
BC BU,43.554323607427065,148
Quebec BU,43.921657657657654,278
GTA BU,48.213860759493684,81
Prairies BU,49.00383154417836,194
Edmonton BU,49.32854988399073,118
Central Ontario BU,51.66308625336927,123


4. Cost Driver Analysis

We analyze cost structure for low-margin routes to identify primary drivers.

In [0]:
low_margin = df.filter(col("gross_margin_pct") < 15)

display(
    low_margin.select(
        (col("total_fuel_cost")/col("total_cost")).alias("fuel_pct"),
        (col("total_labour_cost")/col("total_cost")).alias("labour_pct"),
        (col("total_disposal_cost")/col("total_cost")).alias("disposal_pct"),
        (col("total_maintenance_cost")/col("total_cost")).alias("maintenance_pct"),
        (col("total_admin_cost")/col("total_cost")).alias("admin_pct")
    ).agg(
        avg("fuel_pct"),
        avg("labour_pct"),
        avg("disposal_pct"),
        avg("maintenance_pct"),
        avg("admin_pct")
    )
)

avg(fuel_pct),avg(labour_pct),avg(disposal_pct),avg(maintenance_pct),avg(admin_pct)
0.043708689623374375,0.1545225827050993,0.7456025199559239,0.0152455540083228,0.040920653707279674


5. 3-Year Performance Trend
We evaluate whether profitability is improving or declining over time.

In [0]:

display(
    df.groupBy("year")
      .agg(
          avg("gross_margin_pct").alias("avg_margin"),
          sum("total_profit").alias("total_profit"),
          avg("cost_per_stop").alias("avg_cost_per_stop")
      )
      .orderBy("year")
)

year,avg_margin,total_profit,avg_cost_per_stop
2022,44.47314690026954,1.4308082979999999E7,28.89389824797844
2023,44.779346205962064,1.53556315E7,29.893523035230356
2024,45.44297501643658,1.6522713940000001E7,30.722274819197903


## Key Findings

- Identify worst-performing regions and BUs
- Labour cost is the primary driver of low margin routes
- Certain areas consistently underperform due to inefficiency in stop density
- Profitability shows either improvement or decline over time (based on results)

## Recommendation
Optimize route clustering and rebalance stop density in high-cost regions.